Encoder Only: Finding the Most Influential Word of an Idiom

In [3]:
import pandas as pd
from sentence_transformers import SentenceTransformer
# Read the CSV file
df = pd.read_csv('../large_idiom_set_fr_eng.csv')

# Get first idiom
first_idiom = "appeler un chat un chat"

# Filter for first idiom only
df_first = df[df['contains_idioms'] == first_idiom].copy()

# Original texts
df_first['french_text'] = df_first['original_text']
df_first['english_text'] = df_first['text']

# Get the individual words from the idiom
idiom_words = first_idiom.split()  

# For each word, delete it from the text
for i, word in enumerate(idiom_words):
    df_first[f'french_{i+1}stWordDeleted'] = df_first['original_text'].str.replace(
        f' {word}\\b', '', case=False, regex=True
    )

df = df_first

# Verify
print(f"Original text: {df['original_text'].iloc[0]}")
print(f"Word '{idiom_words[0]}' deleted: {df['french_1stWordDeleted'].iloc[0]}")
print(f"Word '{idiom_words[1]}' deleted: {df['french_2stWordDeleted'].iloc[0]}")
print(f"Word '{idiom_words[2]}' deleted: {df['french_3stWordDeleted'].iloc[0]}")


Original text: Nous appellerons un chat un chat.
Word 'appeler' deleted: Nous appellerons un chat un chat.
Word 'un' deleted: Nous appellerons chat chat.
Word 'chat' deleted: Nous appellerons un un.


In [4]:
# Load multilingual embedding model (works well for English and French)
model = SentenceTransformer('sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2')
df['idiom_id'] = range(len(df))

# Get all french column names (original + deleted versions)
french_columns = ['french_text'] + [col for col in df.columns if col.startswith('french_') and 'Deleted' in col]

# Generate embeddings for all French variations
all_dfs = []

for col in french_columns:
    print(f"Generating embeddings for {col}...")
    embeddings = model.encode(df[col].tolist(), show_progress_bar=True)
    
    temp_df = pd.DataFrame({
        'text': df[col],
        'language': 'French',
        'variation': col,
        'idiom_id': df['idiom_id'],
        'embedding': list(embeddings)
    })
    all_dfs.append(temp_df)

# Generate embeddings for English text (unchanged)
print("Generating English embeddings...")
english_embeddings = model.encode(df['english_text'].tolist(), show_progress_bar=True)

english_df = pd.DataFrame({
    'text': df['english_text'],
    'language': 'English',
    'variation': 'english_text',
    'idiom_id': df['idiom_id'],
    'embedding': list(english_embeddings)
})

all_dfs.append(english_df)

# Combine all DataFrames
combined_df = pd.concat(all_dfs, ignore_index=True)

Generating embeddings for french_text...


Batches:   0%|          | 0/2 [00:00<?, ?it/s]

Generating embeddings for french_1stWordDeleted...


Batches:   0%|          | 0/2 [00:00<?, ?it/s]

Generating embeddings for french_2stWordDeleted...


...

Batches:   0%|          | 0/2 [00:00<?, ?it/s]

Generating embeddings for french_5stWordDeleted...


Batches:   0%|          | 0/2 [00:00<?, ?it/s]

Generating English embeddings...


Batches:   0%|          | 0/2 [00:00<?, ?it/s]

In [5]:
# Reduce embeddings to 2D using UMAP
from umap import UMAP
import numpy as np

print("Reducing embeddings to 2D...")
all_embeddings = np.vstack(combined_df['embedding'].values)
reducer = UMAP(n_components=2, random_state=42)
embeddings_2d = reducer.fit_transform(all_embeddings)

# Add 2D coordinates to dataframe
combined_df['x'] = embeddings_2d[:, 0]
combined_df['y'] = embeddings_2d[:, 1]

# Save to CSV for online visualization
output_df = combined_df[['text', 'language', 'variation', 'idiom_id', 'x', 'y']].copy()
output_df.to_csv('embeddings_2d.csv', index=False)
print("Saved embeddings to 'embeddings_2d.csv'")

# Also save with full embeddings in parquet format
combined_df.to_parquet('embeddings_full.parquet')
print("Saved full embeddings to 'embeddings_full.parquet'")

print(f"\nDataset info: {len(output_df)} points total")
print(f"English: {len(output_df[output_df['language']=='English'])} points")
print(f"French: {len(output_df[output_df['language']=='French'])} points")
print(f"French variations: {output_df[output_df['language']=='French']['variation'].nunique()}")

Reducing embeddings to 2D...


C:\Users\benja\AppData\Roaming\Python\Python312\site-packages\umap\umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


Saved embeddings to 'embeddings_2d.csv'
Saved full embeddings to 'embeddings_full.parquet'

Dataset info: 287 points total
English: 41 points
French: 246 points
French variations: 6


In [6]:
import numpy as np
import pandas as pd

# 1. Get English baseline (unchanged)
eng_df = combined_df[combined_df['language'] == 'English'].copy()

# 2. Get all French variations
fr_variations = combined_df[combined_df['language'] == 'French'].copy()

# 3. Merge English with each French variation
paired_df = pd.merge(
    eng_df[['idiom_id', 'text', 'x', 'y']], 
    fr_variations[['idiom_id', 'text', 'variation', 'x', 'y']], 
    on='idiom_id', 
    suffixes=('_eng', '_fr')
)

# 4. Calculate Euclidean distance in 2D UMAP space
paired_df['visual_distance'] = np.sqrt(
    (paired_df['x_eng'] - paired_df['x_fr'])**2 + 
    (paired_df['y_eng'] - paired_df['y_fr'])**2
)

# 5. Print Statistics per variation
print("=== Distance Statistics by French Variation ===\n")
for variation in paired_df['variation'].unique():
    var_df = paired_df[paired_df['variation'] == variation]
    print(f"{variation}:")
    print(f"  Average distance: {var_df['visual_distance'].mean():.4f}")
    print(f"  Min distance: {var_df['visual_distance'].min():.4f}")
    print(f"  Max distance: {var_df['visual_distance'].max():.4f}\n")

# 6. Find which word deletion causes largest distance change
print("\n=== Most Influential Word (Largest Distance from Original) ===")
baseline_dist = paired_df[paired_df['variation'] == 'french_text']['visual_distance'].mean()
for variation in paired_df['variation'].unique():
    if variation != 'french_text':
        var_dist = paired_df[paired_df['variation'] == variation]['visual_distance'].mean()
        change = abs(var_dist - baseline_dist)
        print(f"{variation}: Δ = {change:.4f}")

=== Distance Statistics by French Variation ===

french_text:
  Average distance: 11.0790
  Min distance: 0.1689
  Max distance: 18.7186

french_1stWordDeleted:
  Average distance: 11.0558
  Min distance: 0.1628
  Max distance: 18.6975

french_2stWordDeleted:
  Average distance: 12.0318
  Min distance: 0.2277
  Max distance: 21.3613

french_3stWordDeleted:
  Average distance: 7.1094
  Min distance: 0.0623
  Max distance: 18.6286

french_4stWordDeleted:
  Average distance: 12.0053
  Min distance: 0.2018
  Max distance: 21.2989

french_5stWordDeleted:
  Average distance: 7.1144
  Min distance: 0.0433
  Max distance: 18.6334


=== Most Influential Word (Largest Distance from Original) ===
french_1stWordDeleted: Δ = 0.0232
french_2stWordDeleted: Δ = 0.9528
french_3stWordDeleted: Δ = 3.9696
french_4stWordDeleted: Δ = 0.9263
french_5stWordDeleted: Δ = 3.9647


In Terminal : embedding-atlas Code3MotPivot/embeddings_full.parquet --x x --y y --text text